# Cleveland Heart Disease — Data Cleaning

Loads the processed Cleveland subset (297 of 303 patients after dropping
rows with missing `ca`/`thal`), casts nominal features to categorical
dtype, checks for outliers, and derives a binary disease-presence label.

In [1]:
import numpy as np
import pandas as pd
import sklearn

## Load the Cleveland dataset

Reads `processed.cleveland.data` (confirmed by UCI's own `WARNING` file to
be the non-corrupted file — the raw `cleveland.data` was damaged during a
node migration) into the standard 14-column layout, mapping the `?`
missing-value marker to `NaN`.

In [2]:
COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
    ]

df_cleveland = pd.read_csv(
    "heart+disease/processed.cleveland.data", header=None, names=COLUMN_NAMES, na_values="?",
    )
df_cleveland.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [3]:
df_cleveland.shape

(303, 14)

## Inspect the data

Check shape, dtypes, missing values, and duplicates before cleaning.

In [4]:
print(df_cleveland.dtypes)
print()
print("Missing values per column:")
print(df_cleveland[["ca", "thal"]].isna().sum())
print()
print("number of duplicated rows:")
print(df_cleveland.duplicated().sum())

age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
dtype: object

Missing values per column:
ca      4
thal    2
dtype: int64

number of duplicated rows:
0


## Remove missing values

Drop rows with missing `ca`/`thal` values (mapped from the `?` marker).

In [5]:
df_cleveland = df_cleveland.replace('?', pd.NA)
df_cleveland = df_cleveland.dropna()

print("Missing values per column:")
print(df_cleveland[["ca", "thal"]].isna().sum())
print(df_cleveland.shape)


Missing values per column:
ca      0
thal    0
dtype: int64
(297, 14)


## Encode nominal features as categorical

`cp`, `restecg`, `slope`, and `thal` are nominal codes (chest pain type,
resting ECG result, ST-segment slope, thalassemia result), not continuous
magnitudes, so they're cast to pandas `category` dtype. `sex`, `fbs`, and
`exang` are also categorical (binary) but are kept as numeric 0/1 here.

In [6]:
for col in ["cp", "restecg", "slope", "thal"]:
    df_cleveland[col] = df_cleveland[col].astype("category")

## Outlier check

Per the proposal (§3.2), outliers are retained unless clearly erroneous.
This screens for physiologically impossible values (which would indicate
data-entry errors) and separately reports how many values sit outside the
IQR fences per continuous feature — for visibility only, nothing is
removed.

In [7]:


impossible = {
    "trestbps <= 0": (df_cleveland["trestbps"] <= 0).sum(),
    "chol <= 0": (df_cleveland["chol"] <= 0).sum(),
    "thalach <= 0": (df_cleveland["thalach"] <= 0).sum(),
    "age <= 0": (df_cleveland["age"] <= 0).sum(),
    "oldpeak < 0": (df_cleveland["oldpeak"] < 0).sum(),
}
print("impossible values (which means data-entry errors):")
for check, count in impossible.items():
    print(f"  {check}: {count}")

print()
print("IQR-based outlier counts (to inspect only, not removed):")
for col in ["age", "trestbps", "chol", "thalach", "oldpeak"]:
    q1, q3 = df_cleveland[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df_cleveland[col] < lower) | (df_cleveland[col] > upper)).sum()
    print(f"  {col}: {n_outliers} values outside [{lower:.1f}, {upper:.1f}]")

impossible values (which means data-entry errors):
  trestbps <= 0: 0
  chol <= 0: 0
  thalach <= 0: 0
  age <= 0: 0
  oldpeak < 0: 0

IQR-based outlier counts (to inspect only, not removed):
  age: 0 values outside [28.5, 80.5]
  trestbps: 9 values outside [90.0, 170.0]
  chol: 5 values outside [113.5, 373.5]
  thalach: 1 values outside [83.5, 215.5]
  oldpeak: 5 values outside [-2.4, 4.0]


## Inspect the target's severity levels

num is the angiographic diagnosis on a 0–4 severity scale (0 = no
significant narrowing, 1–4 = increasing degrees of narrowing across major
vessels). This checks the class counts before collapsing it to a binary
label below.

In [8]:
df_cleveland["num"].value_counts()

num
0    160
1     54
2     35
3     35
4     13
Name: count, dtype: int64

## Binarise the target

Collapses `num` into a binary disease presence/absence label: `False` (0)
= no significant narrowing, `True` (1–4 combined) = disease present.
Stored in a new `check` column; `num` itself is kept in the dataframe
rather than dropped.

In [9]:
df_cleveland["check"] = (df_cleveland["num"] > 0).astype("category")
df_cleveland["check"].value_counts()

check
False    160
True     137
Name: count, dtype: int64

## Save the cleaned dataset

Writes the cleaned dataframe to new file `heart+disease/cleveland_clean.csv`.

In [10]:
df_cleveland.to_csv("heart+disease/cleveland_clean.csv", index=False)

# check the saved cleaned data

In [ ]:
df = pd.read_csv("heart+disease/cleveland_clean.csv")


print(df[["age","trestbps","chol","thalach","oldpeak"]].describe())
print()
for c in ["cp","restecg","slope","thal","ca","sex","fbs","exang"]:
    print(c, df[c].value_counts().to_dict())

# cp: chest pain type
# trestbps: resting blood pressure
# chol: serum cholesterol
# fbs:fasting blood sugar
# restecg: resting electrocardiographic results
# thalach: maximum heart rate achieved
# exang: exercise-induced angina
# thal: thalassemia

print(df["check"].value_counts(normalize=True))

print()
print(df.groupby("sex")["check"].agg(["size", "sum", "mean"]))
#  sum=positive num

              age    trestbps        chol     thalach     oldpeak
count  297.000000  297.000000  297.000000  297.000000  297.000000
mean    54.542088  131.693603  247.350168  149.599327    1.055556
std      9.049736   17.762806   51.997583   22.941562    1.166123
min     29.000000   94.000000  126.000000   71.000000    0.000000
25%     48.000000  120.000000  211.000000  133.000000    0.000000
50%     56.000000  130.000000  243.000000  153.000000    0.800000
75%     61.000000  140.000000  276.000000  166.000000    1.600000
max     77.000000  200.000000  564.000000  202.000000    6.200000

cp {4.0: 142, 3.0: 83, 2.0: 49, 1.0: 23}
restecg {0.0: 147, 2.0: 146, 1.0: 4}
slope {1.0: 139, 2.0: 137, 3.0: 21}
thal {3.0: 164, 7.0: 115, 6.0: 18}
ca {0.0: 174, 1.0: 65, 2.0: 38, 3.0: 20}
sex {1.0: 201, 0.0: 96}
fbs {0.0: 254, 1.0: 43}
exang {0.0: 200, 1.0: 97}
check
False    0.538721
True     0.461279
Name: proportion, dtype: float64

     size  sum      mean
sex                     
0.0    96   25 